[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Linear Algebra &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


**1.** Blend Bergen, Oslo and Tromso with the weights 0.2, 0.3 and 0.5 using `@`, and print the blend's first three hours, rounded to two places.

In [2]:
other_weights = np.array([0.2, 0.3, 0.5])

print((other_weights @ readings[[0, 1, 3]])[:3].round(2))

[-5.12 -6.58 -6.58]


More weight on Tromso makes the blend colder than the worked example's.

**2.** Multiply `[[2, 0], [0, 3]]` and `[[1, 4], [5, 6]]` as arrays, with `@` and with `*`, and print both results.

In [3]:
scale = np.array([[2, 0], [0, 3]])
values = np.array([[1, 4], [5, 6]])

print(scale @ values)
print(scale * values)

[[ 2  8]
 [15 18]]
[[ 2  0]
 [ 0 18]]


With `@`, the first array doubles the first row of the second and triples the second row. With `*`, the zeros remove the values in their places.

**3.** Fit a straight line to Bergen's monthly means for January to July, months 0 to 6, with `np.linalg.lstsq`, and print the intercept and the slope, rounded to two places.

In [4]:
DAYS_IN_MONTH = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
months = np.repeat(np.arange(12), [24 * length for length in DAYS_IN_MONTH])
monthly = np.array([readings[0][months == month].mean() for month in range(12)])

first_half = np.arange(7.0)
line = np.column_stack([np.ones(7), first_half])

print(np.linalg.lstsq(line, monthly[:7], rcond=None)[0].round(2))

[-2.19  3.32]


Bergen warmed by about 3.3 degrees a month from January to July, along the straight line that comes closest to those seven means.

**4.** Fit the model in `design` to Oslo's readings, row 1, and print the mean and the seasonal and daily swings, rounded to two places.

In [5]:
hour_index = np.arange(HOURS)
year_angle = 2 * np.pi * hour_index / HOURS
day_angle = 2 * np.pi * (hour_index % 24) / 24
design = np.column_stack([
    np.ones(HOURS),
    np.cos(year_angle), np.sin(year_angle),
    np.cos(day_angle), np.sin(day_angle),
])

oslo_fit = np.linalg.lstsq(design, readings[1], rcond=None)[0]

print(oslo_fit[0].round(2))
print(np.hypot(oslo_fit[1], oslo_fit[2]).round(2), np.hypot(oslo_fit[3], oslo_fit[4]).round(2))

6.5
9.0 3.0


Oslo has the same swings as Bergen around a lower mean.

**5.** Fit the model to Svalbard's readings, leaving out the missing hours with a mask on both `design` and the readings, and print the date and time at which the fitted seasonal swing peaks.

In [6]:
reported = ~np.isnan(readings[2])
svalbard_fit = np.linalg.lstsq(design[reported], readings[2][reported], rcond=None)[0]
peak = np.arctan2(svalbard_fit[2], svalbard_fit[1]) % (2 * np.pi) / (2 * np.pi) * HOURS

print(START + timedelta(hours=round(peak)))

2025-07-19 04:00:00


Svalbard's season peaks at the same hour as Bergen's, because the formula times every station's seasons alike.

---

&#8592; **Back to:** [Linear Algebra](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/13-linear-algebra.ipynb)  &nbsp;&middot;&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
